# Customer Churn Modeling

## Objective
Build and evaluate classification models to predict customer churn using the cleaned dataset.

This notebook covers all modeling tasks:

- ? Train/test split and feature scaling (as needed)
- ? Model training: Logistic Regression, Random Forest, Gradient Boosting
- ? Cross-validation and hyperparameter tuning
- ? Compare models using AUC-ROC and accuracy
- ? Interpret results and select the best model for deployment

**Goal**: Build a predictive model that accurately forecasts customer churn and provides actionable insights for retention.

# Setup

## Imports 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score




# Data Preparation

## Data Loading

In [ ]:
df = pd.read_csv('datasets/churn_cleaned.csv')
df.head()

In [ ]:
df.info()

## Feature and Target Separation

In [ ]:
X = df.drop('churn', axis=1)
y = df['churn']

## Train/Test Split

- Stratification ensures that the distribution of the target variable (churn) is approximately the same in both the training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
# Check the stratification
# Check original class distribution
print("Full dataset:")
print(y.value_counts(normalize=True))

# Check train set
print("\nTrain set:")
print(y_train.value_counts(normalize=True))

# Check test set
print("\nTest set:")
print(y_test.value_counts(normalize=True))

## Scale Numeric Features

In [ ]:
numeric_features = ['monthlycharges', 'totalcharges', 'tenure_months']
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

# Modeling

## 4.1 Model Evaluation Function

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name="Model"):
    model.fit(X_train, y_train) # Train the model (including best params, if it's from GridSearchCV)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Evaluate the model on the test set
    roc_auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n?? {model_name}")
    print("Accuracy:", round(acc, 4))
    print("ROC AUC :", round(roc_auc, 4))
    
    return {"model": model_name, "accuracy": acc, "roc_auc": roc_auc}


## Baseline Model: Logistic Regression

In [ ]:
# Initialize and train model
model_results = []

lr = LogisticRegression(max_iter=1000, random_state=42)

model_results.append(evaluate_model(lr, X_train, X_test, y_train, y_test, "Logistic Regression"))

## Compare with Other Models

- Decision Tree

- Random Forest

- Gradient Boosting (optional)



### DecisionTree Classifier

In [ ]:
# Create the baseline DT model
dt = DecisionTreeClassifier(random_state=42)
model_results.append(evaluate_model(dt, X_train, X_test, y_train, y_test, "Decision Tree"))

In [ ]:
# Tune the DT model
# Define hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Set up GridSearchCV
grid_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

# Fit on training data
grid_dt.fit(X_train, y_train)

# Best model from grid
best_dt = grid_dt.best_estimator_

# Evaluate the best model
model_results.append(evaluate_model(best_dt, X_train, X_test, y_train, y_test, "Decision Tree (Tuned)"))


## RandomForest Classifier

In [ ]:
# Define the hyperparameter grid
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Initialize GridSearchCV
grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1  # shows progress
)

# Fit on training data
grid_rf.fit(X_train, y_train)

# Get the best model
best_rf = grid_rf.best_estimator_

In [ ]:
model_results.append(evaluate_model(
    best_rf, X_train, X_test, y_train, y_test, "Random Forest (Tuned)"
))

In [ ]:
print("Best hyperparameters:", grid_rf.best_params_)

### Feature Importance Interpretation (What the Model Thinks Matters Most)

In [ ]:
# Examine at the feature importance
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns)

# Sort and plot top features
top_n = 10  # show top 10
importances.sort_values(ascending=True).tail(top_n).plot(kind='barh', figsize=(8, 6))
plt.title("Top Feature Importances (Random Forest)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

After training the Random Forest model, we looked at which features it relied on the most to make its predictions. Not surprisingly, **tenure** came out on top, the shorter someone has been with the company, the more likely they are to churn. This matches what we saw earlier in our EDA and statistical testing.

**Total charges** and **monthly charges** were also important, suggesting that how much a customer has paid (or is currently paying) affects their likelihood to leave.

Other standout features included contract type and internet service; customers on longer contracts and those not using fiber optic internet seemed less likely to churn, which also makes sense from a business perspective.

Looking at these results helps confirm that the model is picking up on patterns that align with what we already know ? and gives us ideas for how to better support at-risk customers.

## XGBoost Classifier

In [ ]:
# Define hyperparameter grid
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 10],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

# Set up GridSearchCV
grid_xgb = GridSearchCV(
    estimator=XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid=param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Fit the model
grid_xgb.fit(X_train, y_train)

# Get the best model
best_xgb = grid_xgb.best_estimator_

# Evaluate
model_results.append(evaluate_model(best_xgb, X_train, X_test, y_train, y_test, "XGBoost (Tuned)"))

### Feature Importance for XGBoost

In [ ]:
# Get feature importances from the trained model
xgb_importances = pd.Series(best_xgb.feature_importances_, index=X_train.columns)

# Sort and plot
top_n = 10
xgb_importances.sort_values(ascending=True).tail(top_n).plot(kind='barh', figsize=(8, 6))
plt.title("Top Feature Importances (XGBoost)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

### What XGBoost Tells Us About Churn

After tuning XGBoost, we looked at which features the model relied on most when making predictions. The strongest signal came from **contract type**, especially whether a customer had a **two-year contract**. Customers with long-term contracts were much less likely to churn ? which makes sense given the commitment involved.

Another top feature was whether the customer had **fiber optic internet**, which often comes with higher monthly charges ? possibly linked to higher churn. XGBoost also picked up on whether customers had **no internet service at all**, which might reflect low usage or auto-cancellation situations.

Interestingly, **tenure** and **total charges**, which were the top features in Random Forest, were less important here. XGBoost focused more on **account structure** (contract and service choices) rather than usage behavior.

---

### Comparing XGBoost and Random Forest

Both models identified meaningful patterns, they just approached the problem from different angles. 

This gives us a more well-rounded understanding of what drives churn: both **how long someone has been with the company** and **what kind of plan or services they?re using**.

# Evaluation

In [ ]:
# Create DataFrame from model_results list
results_df = pd.DataFrame(model_results)

# Set model name as index
results_df.set_index("model", inplace=True)

# Sort by accuracy
results_df = results_df.sort_values("roc_auc", ascending=False)

# Display
print("Model Comparison Summary (Sorted by ROC AUC):")
display(results_df.round(4))  # rounds metrics for cleaner display

We compared five models using both accuracy and ROC AUC, with a focus on identifying customer churn effectively:

**XGBoost** delivered the best results overall, with the highest ROC AUC (0.8956), showing it was the most effective at separating churners from non-churners. Random Forest and the tuned Decision Tree also performed well, with strong AUC scores and solid accuracy.

Even Logistic Regression performed competitively, confirming that some of the key patterns in churn are relatively linear and predictable.

The untuned Decision Tree was the weakest performer ? reinforcing the importance of hyperparameter tuning and model selection when working with classification problems like churn.

# Final Recommendation

After testing and tuning several classification models, we recommend using the **tuned XGBoost model** to predict customer churn.

This model achieved the highest performance overall, with an **accuracy of 85.7%** and a **ROC AUC score of 0.896**, making it the most effective at distinguishing between customers who are likely to churn and those who are not.

XGBoost also offered interpretable insights into which features matter most ? highlighting contract type, internet service, and tenure as key drivers of churn. These findings support earlier analysis and align with known business logic.

**Why XGBoost?**
- Strong predictive performance (highest AUC + accuracy)
- Tuned to balance model complexity and generalization
- Interpretable outputs (feature importance)
- Performs well with mixed data types and slight imbalance

This model would be a reliable foundation for a churn-risk dashboard, outreach automation, or targeted retention campaigns.